# GPU Batching with TMol

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uw-ipd/tmol/blob/kdidi/sphinx-docs/docs/tutorial/02_gpu_batching.ipynb)

TMol gains throughput by evaluating many structures together in one `PoseStack`. This notebook puts that capability near the start of the tutorial path and benchmarks scoring with synchronization placed correctly around CUDA timings. Scores are used here as batching outputs; [Scoring and Analysis](03_scoring_and_analysis.ipynb) next explains their term-by-term scientific interpretation.

> **Prerequisite.** Complete [Working with TMol](01_working_with_tmol.ipynb) first; this notebook reuses its `PoseStack`, CIF-input, and explicit-device vocabulary.

## Goals

- Build repeated and heterogeneous batches with `PoseStackBuilder.from_poses`.
- Score several complete CIF structures in one call and click through labeled results.
- Record enough hardware and software metadata to interpret timings.
- Measure latency, throughput, and peak CUDA memory after warmup.
- Provide a tiny CPU fallback and explain practical chunking.
- Distinguish TMol tensor batching from Chapter 16 job parallelism.

> **GPU recommended, not required.** CUDA is needed for representative acceleration and memory measurements. The same code uses tiny batch sizes on CPU so the notebook remains runnable.

## Setup

The benchmark fixes random seeds, discovers the checked-in 1UBQ fixture, and uses the same device for the pose, score function, and generated batches.

The controlled scaling benchmark uses only chain A residues 1–20. This small supported fragment keeps the CPU documentation path practical and makes repeated-pose batching cheap enough to demonstrate; it is not presented as a biological subsystem. Because slicing creates a new chain end, TMol assigns the appropriate terminal block type during import. The heterogeneous example later in the notebook returns to complete checked-in structures.

In [ ]:
try:
    import google.colab  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    from urllib.request import urlopen

    exec(
        urlopen(
            "https://raw.githubusercontent.com/uw-ipd/tmol/"
            "kdidi/sphinx-docs/docs/tutorial/colab_setup.py"
        ).read(),
        globals(),
    )
    setup_colab(
        [
            "tmol/tests/data/cif/1UBQ.cif",
            "tmol/tests/data/cif/1R21.cif",
            "tmol/tests/data/cif/1BL8.cif",
        ]
    )

In [ ]:
from contextlib import redirect_stderr, redirect_stdout
from io import StringIO
from pathlib import Path
import platform
from time import perf_counter
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from biotite.structure.io import load_structure

import tmol
from tmol import beta2016_score_function
from tmol.io.pose_stack_from_biotite import pose_stack_from_biotite
from tmol.pose.pose_stack_builder import PoseStackBuilder

SEED = 20260807
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
repo_root = Path.cwd()
if not (repo_root / "tmol/tests/data/cif/1UBQ.cif").exists():
    repo_root = Path(tmol.__file__).resolve().parents[1]
cif_path = repo_root / "tmol/tests/data/cif/1UBQ.cif"

atom_array = load_structure(str(cif_path), model=1, include_bonds=True)
protein_slice = atom_array[(atom_array.chain_id == "A") & (atom_array.res_id <= 20)]
pose_diagnostics = StringIO()
try:
    with redirect_stdout(pose_diagnostics), redirect_stderr(pose_diagnostics):
        single_pose = pose_stack_from_biotite(protein_slice, device, no_optH=True)
except Exception:
    print(pose_diagnostics.getvalue())
    raise
score_function = beta2016_score_function(device)


def show_table(frame):
    """Use sortable tables in rendered docs, with a pandas fallback."""
    try:
        from itables import show
    except ImportError:
        return display(frame)
    return show(frame)


print(f"benchmark device: {device}; input: {cif_path.name}")

## Record benchmark metadata first

Latency numbers are not portable without the PyTorch/TMol versions, operating system, CUDA runtime, and GPU identity. Record these before timing. Peak memory below is CUDA allocator memory, not total process or driver memory.

In [ ]:
metadata = {
    "platform": platform.platform(),
    "python": platform.python_version(),
    "tmol": tmol.__version__,
    "torch": torch.__version__,
    "device": str(device),
    "cuda_runtime": torch.version.cuda,
}
if device.type == "cuda":
    props = torch.cuda.get_device_properties(device)
    metadata.update(
        {
            "gpu_name": props.name,
            "compute_capability": f"{props.major}.{props.minor}",
            "gpu_memory_GiB": props.total_memory / 2**30,
        }
    )
metadata_frame = pd.DataFrame.from_dict(metadata, orient="index", columns=["value"])
show_table(metadata_frame.reset_index(names="property"))

## Build batches explicitly

`PoseStackBuilder.from_poses` concatenates compatible pose stacks and repacks their tensors on the requested device. Repeating the same pose is useful for a controlled throughput demonstration; real workloads normally batch distinct but chemistry-compatible structures.

In [ ]:
preview_batch = PoseStackBuilder.from_poses([single_pose] * 4, device)
print("single coords:", tuple(single_pose.coords.shape))
print("batch coords: ", tuple(preview_batch.coords.shape))
print("batch poses:  ", preview_batch.n_poses)

## Score a heterogeneous CIF batch

Repeating one pose isolates batching overhead, but production batches usually contain different structures. Here three complete, checked-in CIF structures are converted independently, combined into one `PoseStack`, scored in one call, and exposed through a structure selector. The batch pads its block and atom dimensions to the largest member, so every pose occupies the batch's fixed rectangular layout; grouping similarly sized structures reduces wasted storage and work.

This is the core GPU workflow: one tensor operation returns one weighted total per structure, while the label table and switcher preserve every input's identity. These beta2016 totals are score-function units, not physical energies. Because the proteins are unrelated and differ in size and composition, their absolute totals are **not scientifically comparable**; this table demonstrates batch indexing and labeling, not a ranking.

In [ ]:
structure_specs = {
    "1UBQ — ubiquitin": "1UBQ.cif",
    "1R21": "1R21.cif",
    "1BL8": "1BL8.cif",
}
individual_poses = {}
for label, filename in structure_specs.items():
    structure = load_structure(
        str(repo_root / "tmol" / "tests" / "data" / "cif" / filename),
        model=1,
        include_bonds=True,
    )
    pose_diagnostics = StringIO()
    try:
        with redirect_stdout(pose_diagnostics), redirect_stderr(pose_diagnostics):
            individual_poses[label] = pose_stack_from_biotite(
                structure, device, no_optH=True
            )
    except Exception:
        print(pose_diagnostics.getvalue())
        raise

heterogeneous_batch = PoseStackBuilder.from_poses(
    list(individual_poses.values()), device
)
heterogeneous_scorer = score_function.render_whole_pose_scoring_module(
    heterogeneous_batch
)
with warnings.catch_warnings(), torch.no_grad():
    warnings.filterwarnings("ignore", message=r"Sparse index lookup.*")
    heterogeneous_scores = heterogeneous_scorer(
        heterogeneous_batch.coords
    ).detach().cpu().numpy()

batch_rows = []
for pose_index, (label, pose) in enumerate(individual_poses.items()):
    batch_rows.append(
        {
            "pose_index": pose_index,
            "structure": label,
            "blocks": pose.max_n_blocks,
            "pose_atoms": pose.max_n_pose_atoms,
            "weighted_score": float(heterogeneous_scores[pose_index]),
        }
    )
batch_frame = pd.DataFrame(batch_rows)
show_table(batch_frame)
display(
    tmol.switchable_view(
        individual_poses,
        notes={
            row["structure"]: (
                f"batch index {row['pose_index']}; "
                f"{row['blocks']} blocks; score {row['weighted_score']:.3f}"
            )
            for row in batch_rows
        },
    )
)

## Benchmark methodology

`render_whole_pose_scoring_module(batch)` creates a scorer for that batch's fixed block, atom, and connectivity layout. Reuse it while only coordinates change within the same layout. Re-render when batch size, membership, padding dimensions, or chemical layout changes; `benchmark_scoring()` therefore renders once for each batch size and excludes that construction from the timed region.

CUDA launches are asynchronous. A valid wall-clock measurement therefore:

1. renders the scorer before timing;
2. runs untimed warmup calls so lazy compilation and caches are not charged to steady state;
3. synchronizes before starting and after finishing each timed call; and
4. reports multiple repeats rather than one launch.

The table's `latency_ms` is median **total latency for one batch call**. `latency_per_pose_ms` divides that total by batch size and is an amortized throughput metric, not the time at which one pose's result becomes independently available. The CPU path uses the same function without CUDA synchronization and deliberately tiny sizes.

In [ ]:
def benchmark_scoring(batch_size, repeats=5, warmup=2):
    batch = PoseStackBuilder.from_poses([single_pose] * batch_size, device)
    scorer = score_function.render_whole_pose_scoring_module(batch)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    with torch.no_grad():
        for _ in range(warmup):
            scorer(batch.coords)
        if device.type == "cuda":
            torch.cuda.synchronize(device)

        elapsed = []
        for _ in range(repeats):
            if device.type == "cuda":
                torch.cuda.synchronize(device)
            start = perf_counter()
            scorer(batch.coords)
            if device.type == "cuda":
                torch.cuda.synchronize(device)
            elapsed.append(perf_counter() - start)

    median_seconds = float(np.median(elapsed))
    peak_memory = (
        torch.cuda.max_memory_allocated(device) / 2**20
        if device.type == "cuda"
        else np.nan
    )
    return {
        "batch_size": batch_size,
        "latency_ms": 1e3 * median_seconds,
        "throughput_poses_s": batch_size / median_seconds,
        "peak_cuda_MiB": peak_memory,
        "repeats": repeats,
    }

batch_sizes = [1, 4, 16, 64] if device.type == "cuda" else [1, 2, 4]
repeats = 5 if device.type == "cuda" else 2
benchmark_frame = pd.DataFrame(
    [benchmark_scoring(size, repeats=repeats) for size in batch_sizes]
)
benchmark_frame["latency_per_pose_ms"] = (
    benchmark_frame["latency_ms"] / benchmark_frame["batch_size"]
)
benchmark_frame["throughput_vs_batch_1"] = (
    benchmark_frame["throughput_poses_s"]
    / benchmark_frame.loc[0, "throughput_poses_s"]
)
benchmark_frame.insert(
    0, "measurement", "CUDA throughput" if device.type == "cuda" else "CPU smoke check"
)
show_table(benchmark_frame)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(
    benchmark_frame["batch_size"],
    benchmark_frame["latency_per_pose_ms"],
    marker="o",
)
axes[0].set(
    xlabel="batch size",
    ylabel="median milliseconds / pose",
    title="Amortized latency",
)
axes[1].plot(
    benchmark_frame["batch_size"],
    benchmark_frame["throughput_poses_s"],
    marker="o",
)
axes[1].set(
    xlabel="batch size", ylabel="poses / second", title="Scoring throughput"
)
for axis in axes:
    axis.grid(alpha=0.3)
fig.suptitle(
    "Measured CUDA batching" if device.type == "cuda" else "CPU smoke check — not GPU scaling",
    fontweight="bold",
)
plt.tight_layout()
plt.show()

**Expected observations.** Over the range where one pose under-fills a GPU, total batch-call latency generally grows more slowly than batch size. That is why the separate amortized metric—total milliseconds divided by poses—falls and throughput rises until compute or memory bandwidth saturates. The total latency does not become a per-pose latency, and batching is not superlinear algorithmic scaling: scoring work still grows with the number of poses.

The published documentation is executed on a CPU runner, where these kernels have little batch parallelism; its small table is deliberately labeled a correctness smoke check, not evidence about CUDA performance. Open the notebook in a Colab GPU runtime for the relevant curve. First-call compilation is intentionally excluded.

## CUDA-only capacity probe

The following cell is genuinely GPU-only and is labeled `gpu-only`. It demonstrates a larger batch and allocator memory. It is skipped cleanly on CPU.

In [ ]:
#| tags: [gpu-only]
# gpu-only: this capacity point is omitted on CPU.
if device.type != "cuda":
    print("Skipped: this cell requires CUDA.")
    gpu_capacity_point = None
else:
    gpu_capacity_point = benchmark_scoring(128, repeats=5, warmup=2)
    show_table(pd.DataFrame([gpu_capacity_point]))
gpu_capacity_point

## Memory, padding, and chunking

A larger batch is not always faster. `PoseStack` pads every member to the batch maxima for atoms and blocks, so one large structure can make all shorter members occupy a larger rectangular layout. Pairwise scoring intermediates may also grow with atom and block dimensions. Padding therefore affects both useful throughput and total memory; it is not merely display metadata.

`peak_cuda_MiB` is the **total peak memory allocated by PyTorch's CUDA allocator** after the counter reset. It includes the live batch/scorer baseline plus allocations made during warmup and timed scoring, but excludes non-PyTorch driver and process memory. It is not a per-pose measurement, and dividing it by batch size does not isolate a pose's footprint because setup tensors, padding, and shared term intermediates are not independent per-pose allocations.

Use that peak as a capacity indicator, leave headroom for compilation and downstream tensors, and split a workload into chunks before allocator pressure causes an out-of-memory error. A practical pattern is: choose a conservative per-device chunk size, build and score one chunk, immediately detach or transfer the small results you need, release chunk references, and continue. Grouping similarly sized structures reduces padding waste.

TMol has no built-in multi-GPU scheduler. One process should normally own one GPU; an external Slurm, Dask, Ray, or multiprocessing layer can shard independent chunks across devices.

## Rosetta comparison: two complementary levels of parallelism

TMol batching vectorizes compatible structures inside one process on one device. PyRosetta Chapter 16 mostly distributes independent jobs, trajectories, or protocols across processes and workers. Job distribution handles heterogeneous or long-running tasks; TMol batching amortizes kernels over tensor-compatible work. An outer scheduler can combine both ideas by assigning one TMol batch stream to each GPU.

Representative Chapter 16 entry points:

- [16.00 Running PyRosetta in Parallel](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.00-Running-PyRosetta-in-Parallel.ipynb) introduces the landscape.
- [16.03 GNU Parallel via Slurm](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.03-GNU-Parallel-Via-Slurm.ipynb) shows process/job distribution.
- [16.04 Dask delayed via Slurm](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.04-dask.delayed-Via-Slurm.ipynb) shows task-graph scheduling.
- [16.06 PyRosettaCluster simple protocol](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.06-PyRosettaCluster-Simple-protocol.ipynb) introduces protocol distribution.

The rendered [PyRosetta notebook index](https://rosettacommons.github.io/PyRosetta.notebooks/) links the remaining Chapter 16 examples and setup material.

## Next: interpret the scores

Batch construction, timing, and capacity planning are now separated from scientific interpretation. Continue to [Scoring and Analysis](03_scoring_and_analysis.ipynb) to decompose weighted totals, inspect residue-pair contributions, and ask comparisons that use a consistent molecular system and protocol.

## Exercises

1. Add interquartile latency alongside the median without timing scorer construction.
2. Increase CUDA batch sizes until throughput plateaus, stopping well before memory exhaustion.
3. Compare batches of similarly sized and mixed-length poses to quantify padding cost.
4. Implement an outer loop that scores a large list in conservative chunks and concatenates detached totals.
5. Design a Slurm array where each task selects one GPU and processes many TMol batches; keep random seeds and metadata per task.

## References

- [TMol repository](https://github.com/uw-ipd/tmol)
- [PyTorch CUDA semantics](https://pytorch.org/docs/stable/notes/cuda.html)
- [PyTorch benchmarking recipe](https://pytorch.org/tutorials/recipes/recipes/benchmark.html)
- [PyRosetta Chapter 16 index](https://rosettacommons.github.io/PyRosetta.notebooks/)
- [GNU Parallel via Slurm notebook](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.03-GNU-Parallel-Via-Slurm.ipynb)
- [Dask via Slurm notebook](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.04-dask.delayed-Via-Slurm.ipynb)